<h1 align = "center">
E-Commerce Data Preparation
</h1>


## 1. Objective

In this notebook, our main goal is to effect the necessary changes we have already identified in the data profiling notebook, which we consider will leave the datasets cleaner and easier to manipulate for our Power BI project. 

We'll be following the data prepration decisions identified in the data preparation markdown. Our philosophy remains:

    Don't "clean" data that isn't dirty. Transform only where there is an analytical reason.

Our preparation will therefore involve:

#### Products
 Standardizing column names, Ensuring correct data types, Validating pricing relationships, Standardizing text, and Potentially creating product-level derived fields
#### Sales
 - Converting dates, Converting time, Creating delivery duration, Creating order-hour variables, Creating coupon-used indicator, Investigating negative total amounts, and Standardizing categorical fields
#### Customers
 - Converting dates, Standardizing categorical fields, Validating customer aggregates, Creating customer tenure-related fields where appropriate, and Removing unnecessary PII from the analytical dataset

## 2. Load Raw Data

It's recommended to start from raw files rather than loading already modified profiling objects.

It ensures reproducibility

In [30]:
import pandas as pd
import numpy as np

products = pd.read_csv("../data/raw/products.csv")
sales = pd.read_csv("../data/raw/sales.csv")
customers = pd.read_csv("../data/raw/customers.csv")

## 3. Standardize Column Names
Although existing names are reasonably clean, we want to make them consistently lower case to make out python, and eventually DAX easier to read.

In [31]:
# First create a function that we'll apply to the data frames
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df

In [32]:
# Now apply the function to our datasets
products = clean_column_names(products)
sales = clean_column_names(sales)
customers = clean_column_names(customers)

Just to confirm our column names have been edited appropriately

In [33]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   product_id        2000 non-null   str    
 1   product_name      2000 non-null   str    
 2   category          2000 non-null   str    
 3   brand             2000 non-null   str    
 4   original_price    2000 non-null   float64
 5   discount_percent  2000 non-null   int64  
 6   discount_amount   2000 non-null   float64
 7   selling_price     2000 non-null   float64
 8   stock_quantity    2000 non-null   int64  
 9   weight_kg         2000 non-null   float64
 10  avg_rating        2000 non-null   float64
 11  total_reviews     2000 non-null   int64  
dtypes: float64(5), int64(3), str(4)
memory usage: 187.6 KB


In [34]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 21 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   order_id            250000 non-null  str    
 1   customer_id         250000 non-null  str    
 2   product_id          250000 non-null  str    
 3   order_date          250000 non-null  str    
 4   order_time          250000 non-null  str    
 5   delivery_date       250000 non-null  str    
 6   quantity            250000 non-null  int64  
 7   unit_price          250000 non-null  float64
 8   order_value         250000 non-null  float64
 9   shipping_cost       250000 non-null  float64
 10  coupon_code         50185 non-null   str    
 11  coupon_discount     250000 non-null  float64
 12  total_amount        250000 non-null  float64
 13  payment_mode        250000 non-null  str    
 14  order_status        250000 non-null  str    
 15  rating              120030 non-null  float64


In [35]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        40000 non-null  str    
 1   customer_name      40000 non-null  str    
 2   gender             40000 non-null  str    
 3   age                40000 non-null  int64  
 4   age_group          40000 non-null  str    
 5   date_of_birth      40000 non-null  str    
 6   email              40000 non-null  str    
 7   phone              40000 non-null  str    
 8   city               40000 non-null  str    
 9   state              40000 non-null  str    
 10  pincode            40000 non-null  int64  
 11  registration_date  40000 non-null  str    
 12  customer_tier      40000 non-null  str    
 13  total_orders       40000 non-null  int64  
 14  total_spent        40000 non-null  float64
dtypes: float64(1), int64(3), str(11)
memory usage: 4.6 MB


## 4. Data Type Conversion

### Sales

In [36]:
#This converts the order date in sales dataset
sales["order_date"] = pd.to_datetime(
    sales["order_date"],
    errors="coerce"
)

#This converts the delivery date in sales dataset
sales["delivery_date"] = pd.to_datetime(
    sales["delivery_date"],
    errors="coerce"
)

### Customers

In [37]:
# This converts date of birth in customers dataset
customers["date_of_birth"] = pd.to_datetime(
    customers["date_of_birth"],
    errors="coerce"
)

# This converts registration date in customers dataset
customers["registration_date"] = pd.to_datetime(
    customers["registration_date"],
    errors="coerce"
)

### Order time

In [38]:
sales["order_time"] = pd.to_datetime(
    sales["order_time"],
    format="%H:%M:%S",
    errors="coerce"
).dt.time

In [39]:
sales["order_hour"] = pd.to_datetime(
    sales["order_time"].astype(str),
    format="%H:%M:%S",
    errors="coerce"
).dt.hour

## 5. Derived Variables
we want to create useful derived variables, including:

 - delivery days and
 - coupon used

### Delivery days

In [40]:
sales["delivery_days"] = (
    sales["delivery_date"] -
    sales["order_date"]
).dt.days

### Coupon Used
Will be useful for Power BI to avoid repeatedly asking whether the `coupon_code` field is blank

In [41]:
sales["coupon_used"] = (
    sales["coupon_code"]
    .notna()
    .map({True: "Yes", False: "No"})
)

### Time Dimensions
We'll create time dimensions, although we'll not bring all of them into Power BI

Instead we'll use a `dim_date` table in Power BI, which we'll let handle most time intelligence

In [42]:
sales["order_year"] = sales["order_date"].dt.year
sales["order_month"] = sales["order_date"].dt.month
sales["order_month_name"] = sales["order_date"].dt.month_name()
sales["order_quarter"] = sales["order_date"].dt.quarter
sales["order_day"] = sales["order_date"].dt.day
sales["order_day_name"] = sales["order_date"].dt.day_name()

### Customer Tenure
We also want to create a customer tenure variable.

But first we want to temporarily merge the registration date into Sales

In [43]:
sales = sales.merge(
    customers[
        ["customer_id", "registration_date"]
    ],
    on="customer_id",
    how="left",
    suffixes=("", "_customer")
)

In [44]:
sales["customer_tenure_days"] = (
    sales["order_date"] -
    sales["registration_date"]
).dt.days

## 6. Remove/Exclude Sensitive Fields

### PII Removal and Analytical Field Selection
The customer dataset contains both personally identifiable information (PII) and fields required for commercial analysis. 

Direct identifiers and unnecessary sensitive fields are excluded from the analytical dataset.

The raw customer dataset is retained separately and is not modified.

In [45]:
# Columns containing direct identifiers or unnecessary personal information
pii_columns = [
    "customer_name",
    "email",
    "phone",
    "date_of_birth",
    "pincode"
]

# Remove PII from the analytical customer dataset
customers = customers.drop(columns=pii_columns)

customers.head()

,customer_id,gender,age,age_group,city,state,registration_date,customer_tier,total_orders,total_spent
0,CUST00000001,male,48,46-55,Patiala,Punjab,2024-01-28,Gold,5,40098.48
1,CUST00000002,male,73,65+,Durgapur,West Bengal,2023-08-25,Platinum,8,63676.24
2,CUST00000003,male,31,26-35,Gurugram,Haryana,2023-08-30,Silver,0,0.00
3,CUST00000004,male,41,36-45,Kolkata,West Bengal,2024-03-11,Platinum,5,288324.84
4,CUST00000005,female,53,46-55,Noida,UP,2024-04-19,Platinum,4,69265.75


In [46]:
# Validation
# Confirming that PII columns have been removed
customers.columns.tolist()

['customer_id',
 'gender',
 'age',
 'age_group',
 'city',
 'state',
 'registration_date',
 'customer_tier',
 'total_orders',
 'total_spent']

In [47]:
# Confirming that the analytical customer ID remains unique
customers["customer_id"].is_unique

True

## 7. Financial Validation

In [48]:
negative_total = sales[
    sales["total_amount"] < 0
].copy()

negative_total[
    [
        "order_id",
        "order_value",
        "shipping_cost",
        "coupon_code",
        "coupon_discount",
        "total_amount",
        "order_status"
    ]
]

,order_id,order_value,shipping_cost,coupon_code,coupon_discount,total_amount,order_status
84204,ORD0000084205,35.57,62.36,DIWALI100,100.0,-2.07,Delivered
182883,ORD0000182884,26.13,52.31,DIWALI100,100.0,-21.56,Delivered
207695,ORD0000207696,26.13,62.56,DIWALI100,100.0,-11.31,Delivered
216234,ORD0000216235,26.13,57.40,DIWALI100,100.0,-16.47,Delivered
249820,ORD0000249821,35.57,52.45,DIWALI100,100.0,-11.98,Cancelled


In [49]:
negative_total.shape

(5, 32)

In [50]:
negative_total["coupon_code"].value_counts(dropna=False)

coupon_code
DIWALI100    5
Name: count, dtype: int64

In [51]:
negative_total["order_status"].value_counts()

order_status
Delivered    4
Cancelled    1
Name: count, dtype: int64

## 8. Export Prepared Data

### Customers

In [52]:
customers.head()

,customer_id,gender,age,age_group,city,state,registration_date,customer_tier,total_orders,total_spent
0,CUST00000001,male,48,46-55,Patiala,Punjab,2024-01-28,Gold,5,40098.48
1,CUST00000002,male,73,65+,Durgapur,West Bengal,2023-08-25,Platinum,8,63676.24
2,CUST00000003,male,31,26-35,Gurugram,Haryana,2023-08-30,Silver,0,0.00
3,CUST00000004,male,41,36-45,Kolkata,West Bengal,2024-03-11,Platinum,5,288324.84
4,CUST00000005,female,53,46-55,Noida,UP,2024-04-19,Platinum,4,69265.75


In [53]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   customer_id        40000 non-null  str           
 1   gender             40000 non-null  str           
 2   age                40000 non-null  int64         
 3   age_group          40000 non-null  str           
 4   city               40000 non-null  str           
 5   state              40000 non-null  str           
 6   registration_date  40000 non-null  datetime64[us]
 7   customer_tier      40000 non-null  str           
 8   total_orders       40000 non-null  int64         
 9   total_spent        40000 non-null  float64       
dtypes: datetime64[us](1), float64(1), int64(2), str(6)
memory usage: 3.1 MB


In [ ]:
customers.to_csv("../data/processed/Dim_Customers.csv",index=False)

### Products

In [55]:
products.head()

,product_id,product_name,category,brand,original_price,discount_percent,discount_amount,selling_price,stock_quantity,weight_kg,avg_rating,total_reviews
0,PROD000001,Samsung Mobile V5,Electronics,Samsung,130561.34,23,30029.11,100532.23,159,2.63,4.49,102
1,PROD000002,HP Laptop V5,Electronics,HP,36494.92,34,12408.27,24086.65,54,4.19,4.32,107
2,PROD000003,boAt Headphones V2,Electronics,boAt,106587.94,42,44766.93,61821.01,638,1.48,4.31,99
3,PROD000004,Noise Watch V7,Electronics,Noise,81275.27,47,38199.38,43075.89,572,3.30,4.42,127
4,PROD000005,Apple iPhone V4,Electronics,Apple,13099.36,43,5632.72,7466.64,865,3.71,4.37,98


In [56]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   product_id        2000 non-null   str    
 1   product_name      2000 non-null   str    
 2   category          2000 non-null   str    
 3   brand             2000 non-null   str    
 4   original_price    2000 non-null   float64
 5   discount_percent  2000 non-null   int64  
 6   discount_amount   2000 non-null   float64
 7   selling_price     2000 non-null   float64
 8   stock_quantity    2000 non-null   int64  
 9   weight_kg         2000 non-null   float64
 10  avg_rating        2000 non-null   float64
 11  total_reviews     2000 non-null   int64  
dtypes: float64(5), int64(3), str(4)
memory usage: 187.6 KB


In [ ]:
products.to_csv("../data/processed/Dim_Products.csv",index=False)

### Sales

In [58]:
sales.head()

,order_id,customer_id,product_id,order_date,order_time,delivery_date,quantity,unit_price,order_value,shipping_cost,...,delivery_days,coupon_used,order_year,order_month,order_month_name,order_quarter,order_day,order_day_name,registration_date,customer_tenure_days
0,ORD0000000001,CUST00014303,PROD001017,2026-06-07,08:20:00,2026-06-10,1,2622.46,2622.46,0.0,...,3,No,2026,6,June,2,7,Sunday,2023-06-29,1074
1,ORD0000000002,CUST00034253,PROD000703,2025-02-04,20:05:00,2025-02-07,1,90258.00,90258.00,0.0,...,3,No,2025,2,February,1,4,Tuesday,2023-12-03,429
2,ORD0000000003,CUST00000421,PROD000318,2026-03-12,14:59:00,2026-03-19,1,3536.50,3536.50,0.0,...,7,No,2026,3,March,1,12,Thursday,2023-06-14,1002
3,ORD0000000004,CUST00026507,PROD000213,2025-01-27,14:33:00,2025-02-02,1,34081.56,34081.56,0.0,...,6,Yes,2025,1,January,1,27,Monday,2023-06-20,587
4,ORD0000000005,CUST00009418,PROD001437,2024-06-30,07:19:00,2024-07-02,2,69021.17,138042.34,0.0,...,2,No,2024,6,June,2,30,Sunday,2023-12-17,196


In [59]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 32 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   order_id              250000 non-null  str           
 1   customer_id           250000 non-null  str           
 2   product_id            250000 non-null  str           
 3   order_date            250000 non-null  datetime64[us]
 4   order_time            250000 non-null  object        
 5   delivery_date         250000 non-null  datetime64[us]
 6   quantity              250000 non-null  int64         
 7   unit_price            250000 non-null  float64       
 8   order_value           250000 non-null  float64       
 9   shipping_cost         250000 non-null  float64       
 10  coupon_code           50185 non-null   str           
 11  coupon_discount       250000 non-null  float64       
 12  total_amount          250000 non-null  float64       
 13  payment_mo

In [ ]:
sales.to_csv("../data/processed/Fact_Sales.csv",index=False)